# Notebook 1: Setup y Primer Agente con Strands Agents SDK

## Objetivos de aprendizaje
- Entender qué es **LLMOps** y por qué es imprescindible en producción
- Conocer el framework **Strands Agents** y sus ventajas frente a alternativas
- Configurar el entorno de desarrollo con Amazon Bedrock
- Construir un agente de customer service **paso a paso**, explicando cada decisión
- Definir herramientas (tools) y system prompts siguiendo buenas prácticas
- Ejecutar consultas y analizar el comportamiento del agente críticamente

## Contexto del curso
TechShop es una tienda online de electrónica. A lo largo de este curso construiremos un **agente de IA** para atención al cliente, y lo **operacionalizaremos** siguiendo el ciclo LLMOps: observabilidad, gestión de prompts, evaluación, guardrails y despliegue.

---

## Parte 0: ¿Qué es LLMOps y por qué importa?

### El problema: del prototipo a producción

En software clásico, un bug produce una **excepción clara**: `NullPointerException`, `404 Not Found`, un test que falla. Con LLMs, los fallos son **semánticos**: el modelo responde con confianza algo incorrecto, inventa información (*hallucination*), o ignora instrucciones del system prompt. **No hay stacktrace.**

> **Dato real:** Según Gartner, más del 30% de los proyectos de IA generativa serán abandonados tras el proof-of-concept antes de finalizar 2025 ([Gartner, julio 2024](https://www.gartner.com/en/newsroom/press-releases/2024-07-29-gartner-says-more-than-30-percent-of-generative-ai-projects-will-be-abandoned-after-proof-of-concept-by-end-of-2025)).

### LLMOps = MLOps adaptado a las particularidades de los LLMs

| Dimensión | MLOps clásico | LLMOps |
|-----------|---------------|--------|
| **Outputs** | Espacio finito (clase, número) | Texto libre, estocástico |
| **Tipo de fallo** | Error medible (F1, RMSE) | Respuesta plausible pero incorrecta |
| **Costo principal** | Entrenamiento | **Inferencia** (pago por token) |
| **Artefacto** | Modelo + datos | Modelo + **prompt** + contexto |
| **Evaluación** | Métricas numéricas | Métricas + **juicio semántico** (LLM-as-Judge) |

### El ciclo LLMOps

```mermaid
flowchart LR
    D[Develop] --> E[Evaluate]
    E --> Dep[Deploy]
    Dep --> O[Observe]
    O -->|Iterate| D

    G[GUARDRAILS]:::guardrail

    style D fill:#4a90d9,color:#fff
    style E fill:#4a90d9,color:#fff
    style Dep fill:#4a90d9,color:#fff
    style O fill:#4a90d9,color:#fff
    style G fill:#e74c3c,color:#fff

    classDef guardrail stroke-dasharray: 5 5
```

*GUARDRAILS actúan de forma transversal en todo el ciclo, protegiendo input y output en tiempo real.*

| Fase | Herramienta en este curso | Notebook |
|------|--------------------------|----------|
| **Observe** | Langfuse Tracing | NB 2 |
| **Develop** (prompts) | Langfuse Prompt Management | NB 3 |
| **Evaluate** | Evaluaciones + LLM-as-Judge | NB 4 |
| **Deploy** (CI/CD) | Quality Gate automatizado | NB 5 |
| **Guardrails** | LLM Guard (input/output safety) | NB 6 |
| **Full Pipeline** | Todas las piezas integradas | NB 7 |

> **Referencia:** [Langfuse - LLMOps Overview](https://langfuse.com/docs) · [AWS Bedrock User Guide](https://docs.aws.amazon.com/bedrock/latest/userguide/)

### ¿Qué es un agente de IA?

Un **agente** es un sistema que combina un LLM con **herramientas** y un **bucle de razonamiento**. A diferencia de un simple chatbot (prompt -> respuesta), el agente puede:

1. **Razonar** sobre qué acciones tomar (planificación)
2. **Ejecutar herramientas** (buscar en BD, llamar APIs, calcular)
3. **Observar** los resultados de las herramientas
4. **Iterar** hasta resolver la tarea del usuario

```mermaid
flowchart TD
    SP[System Prompt<br/>personalidad, reglas] --> LLM
    UQ[User Query] --> LLM[LLM<br/>razonamiento]
    LLM --> DEC{¿Necesita<br/>herramienta?}
    DEC -->|Sí| TC[Tool Call<br/>search, FAQ]
    DEC -->|No| RESP[Respuesta al usuario]
    TC --> TR[Tool Result]
    TR --> LLM
```

> **Referencia:** [Anthropic - Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) — Un excelente recurso sobre patrones de diseño de agentes.

### ¿Por qué Strands Agents SDK?

[**Strands Agents**](https://github.com/strands-agents/sdk-python) es un SDK open-source de AWS para construir agentes de IA con un enfoque **model-driven**: el LLM controla el bucle de ejecución, decidiendo qué herramientas usar y cuándo detenerse. Esto contrasta con frameworks que imponen flujos de control rígidos (grafos, cadenas).

**Principios de diseño de Strands Agents:**
- **Simplicidad:** `Agent = modelo + herramientas + prompt`. Sin abstracciones intermedias innecesarias.
- **Herramientas como funciones Python:** Decorar con `@tool` convierte cualquier función en una herramienta invocable por el agente.
- **Model-driven:** El LLM decide el flujo. No hay grafos de estado ni cadenas pre-definidas.
- **Multi-provider:** Soporte nativo para Amazon Bedrock, Anthropic, OpenAI, Ollama, LiteLLM y más.
- **Observabilidad nativa:** Emisión de spans OpenTelemetry estándar para integrar con Langfuse, Datadog, etc.


> **Referencias:**
> - [Strands Agents SDK — GitHub](https://github.com/strands-agents/sdk-python)
> - [Strands Agents — Documentación](https://strandsagents.com/latest/)
> - [Amazon Bedrock — Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html)

---

## Parte 1: Setup del entorno

### 1.1 Verificar dependencias

Nuestro stack técnico:
- **`strands-agents`**: SDK para crear agentes con herramientas y bucle de razonamiento
- **`boto3`**: SDK de AWS para conectar con Amazon Bedrock (el servicio que hospeda los modelos)
- **`langfuse`**: Plataforma de observabilidad que instrumentaremos en el Notebook 2

> Si esto falla, ejecuta desde la carpeta `notebooks/`: `uv sync` (ver README del proyecto).

In [ ]:
# Verificar dependencias (instaladas via: cd notebooks/ && uv sync)
import importlib.metadata

import boto3

print(f"[OK] strands {importlib.metadata.version('strands-agents')}")
print(f"[OK] boto3 {boto3.__version__}")

### 1.2 Configuración del entorno

Usamos el patrón **dotenv** para gestionar credenciales. Este es un estándar de la industria para separar configuración de código ([12-Factor App, Factor III](https://12factor.net/config)):

- **Nunca hardcodeamos credenciales** en el código fuente
- El archivo `.env` está en `.gitignore` y nunca se commitea
- Variables como `AWS_ACCESS_KEY_ID` se leen en tiempo de ejecución

```bash
# Si aún no lo tienes:
cp ../.env.example .env
# Edita .env con tus credenciales AWS
```

> **Buena práctica LLMOps:** En producción, las credenciales van en servicios de secretos (AWS Secrets Manager, HashiCorp Vault), no en archivos `.env`.

In [ ]:
import os
from dotenv import load_dotenv

# Carga variables de entorno desde .env
load_dotenv()

# Verificar que las variables AWS están configuradas
assert os.getenv("AWS_ACCESS_KEY_ID"), "Falta AWS_ACCESS_KEY_ID en .env"
assert os.getenv("AWS_SECRET_ACCESS_KEY"), "Falta AWS_SECRET_ACCESS_KEY en .env"

print(os.getenv("AWS_ACCESS_KEY_ID"))
print(f"[OK] Región AWS: {os.getenv('AWS_REGION', 'us-east-1')}")
print(f"[OK] Modelo: {os.getenv('MODEL_ID', 'us.anthropic.claude-haiku-4-5-20251001-v1:0')}")

### 1.3 Verificar conexión a Amazon Bedrock

**Amazon Bedrock** es un servicio gestionado de AWS que da acceso a modelos fundacionales (Claude, Nova, Llama, Mistral...) a través de una API unificada.

Strands Agents utiliza la [**Converse API**](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html) de Bedrock, que ofrece:
- **Interfaz unificada** para todos los modelos (no necesitas cambiar código al cambiar de modelo)
- **Tool use nativo**: el modelo puede solicitar llamadas a herramientas como parte de su respuesta
- **Streaming** de respuestas
- **Gestión automática** de turnos de conversación (mensajes de usuario, asistente, herramientas)

In [ ]:
bedrock = boto3.client(
    "bedrock-runtime",
    region_name=os.getenv("AWS_REGION", "us-east-1")
)
print("[OK] Conexión a Bedrock Runtime establecida")

**¿Qué acaba de pasar?** Hemos creado un cliente `bedrock-runtime` que se conecta al servicio de inferencia de Bedrock. Este cliente utilizará nuestras credenciales AWS (del `.env`) para autenticarse. Strands Agents usará este mismo servicio internamente a través de su clase `BedrockModel`.

> **Referencia:** [Boto3 Bedrock Runtime Client](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-runtime.html)

---

## Parte 2: Explorar los datos de TechShop

Antes de construir el agente, necesitamos entender su **dominio**. Las herramientas del agente operarán sobre estos datos:

- **Catálogo de productos** (`catalog.json`): nombre, precio, stock, descripción, categoría
- **Preguntas frecuentes** (`faqs.json`): preguntas y respuestas agrupadas por tema (envíos, devoluciones, garantías...)

> **Buena práctica:** Siempre explora y entiende los datos antes de diseñar el agente. La calidad de los datos determina la calidad de las respuestas — *garbage in, garbage out* aplica especialmente con LLMs.

In [ ]:
import json
from pathlib import Path

# Los datos están en el repositorio
DATA_DIR = Path("../../src/techshop_agent/data")

FAQS_FILE = DATA_DIR / "faqs.json"
CATALOG_FILE = DATA_DIR / "catalog.json"

N_ELEMENTS = 10

# Cargar catálogo
with CATALOG_FILE.open(encoding="utf-8") as f:
    catalog = json.load(f)

print(f"Productos en catálogo: {len(catalog)}")
print(f"   Categorías: {sorted({p['categoria'] for p in catalog})}")
print()
for p in catalog[:N_ELEMENTS]:
    print(f"   • {p['nombre']:20s} | {p['precio']:>8.2f}€ | Stock: {p['stock']}")

In [ ]:
# Cargar FAQs
with FAQS_FILE.open(encoding="utf-8") as f:
    faqs = json.load(f)

print(f"Preguntas frecuentes: {len(faqs)}")
print(f"   Temas: {sorted({faq['tema'] for faq in faqs})}")
print()
for faq in faqs[:N_ELEMENTS]:
    print(f"   [{faq['tema']:12s}] {faq['pregunta']}")

> **Para reflexionar:** Observa la estructura de los datos. ¿Qué limitaciones ves? ¿Cómo afectará a las búsquedas? (búsqueda por substring vs. semántica, productos sin categorías detalladas, FAQs con un solo nivel de profundidad...)

---

## Parte 3: Crear el modelo con `BedrockModel`

La clase [`BedrockModel`](https://strandsagents.com/latest/user-guide/concepts/model-providers/amazon-bedrock/) de Strands Agents encapsula la conexión a Amazon Bedrock y la configuración de inferencia.

### Parámetros clave de inferencia

| Parámetro | Qué controla | Valor típico |
|-----------|-------------|-------------|
| `model_id` | Qué modelo usar en Bedrock | `us.anthropic.claude-haiku-4-5-20251001-v1:0` |
| `region_name` | Región AWS donde está habilitado el modelo | `us-east-1` |
| `temperature` | Aleatoriedad de la respuesta (0 = determinista, 1 = creativo) | 0.1 – 0.3 para agentes |
| `max_tokens` | Máximo de tokens en la respuesta | 1024 (controla coste) |
| `top_p` | Nucleus sampling — probabilidad acumulada | 0.9 – 1.0 |

> **Buena práctica LLMOps:** Para agentes en producción, usa `temperature` baja (0.1–0.3). Queremos respuestas **consistentes y predecibles**, no creativas. La creatividad dificulta la evaluación y el debugging.

> **Referencia:** [Strands Agents — Amazon Bedrock Provider](https://strandsagents.com/latest/user-guide/concepts/model-providers/amazon-bedrock/) · [Bedrock Inference Parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/inference-parameters.html)

In [ ]:
from strands.models import BedrockModel

model = BedrockModel(
    model_id=os.getenv("MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    region_name=os.getenv("AWS_REGION", "us-east-1"),
    temperature=0.3,
    max_tokens=1024,
)

print(f"[OK] Modelo configurado: {model.config.get('model_id', 'Not Set')}")

**¿Qué hace `BedrockModel` internamente?**

1. Crea un cliente `boto3` de `bedrock-runtime` con la región especificada
2. Cuando el agente lo invoque, enviará requests a la [Converse API](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html) con los parámetros configurados
3. Gestiona automáticamente el formato de mensajes, tool definitions, y la respuesta del modelo

Elegimos **Claude Haiku 4.5** porque es el modelo más rápido y económico de la familia Claude, ideal para un agente transaccional de customer service donde la latencia importa.

---

## Parte 4: Definir las herramientas (Tools) del agente

### ¿Cómo funciona "tool calling" (function calling)?

El concepto de **tool use** es fundamental en agentes modernos. Así funciona:

1. **Definición:** Tú defines funciones Python y las decoras con `@tool`. Strands Agents extrae automáticamente el esquema (nombre, docstring, parámetros, tipos) y se lo envía al modelo.
2. **Decisión del LLM:** Cuando el usuario hace una pregunta, el LLM decide si necesita una herramienta basándose en la pregunta y los esquemas disponibles.
3. **Llamada:** Si decide usar una herramienta, el LLM genera un JSON con el nombre de la función y los argumentos. El SDK ejecuta la función localmente.
4. **Resultado:** El resultado de la función se envía de vuelta al LLM, que lo incorpora a su razonamiento para generar la respuesta final.

```mermaid
sequenceDiagram
    participant U as Usuario
    participant LLM as LLM (Claude)
    participant SDK as Strands SDK
    participant T as search_catalog()

    U->>LLM: "¿Tenéis auriculares?"
    LLM->>SDK: tool_use: search_catalog(query="auriculares")
    SDK->>T: Ejecuta función Python
    T-->>SDK: [{"nombre": "AirPods Pro", ...}]
    SDK-->>LLM: Tool result (JSON)
    LLM-->>U: "Sí, tenemos los AirPods Pro por 279€..."
```

### Buenas prácticas para herramientas

| Práctica | Por qué | Ejemplo |
|----------|---------|---------|
| **Docstring descriptivo** | El LLM lo lee para decidir cuándo usar la tool | `"Busca en el catálogo por nombre o categoría"` |
| **Nombres claros** | Ayudan al LLM a elegir la herramienta correcta | `search_catalog` vs `do_stuff` |
| **Tipos anotados** | El SDK genera mejor esquema JSON para el modelo | `query: str` vs `query` |
| **Retorno serializable** | El resultado vuelve al LLM como texto | Devolver JSON string, no objetos complejos |
| **Una responsabilidad** | Cada herramienta hace una cosa bien | Separar búsqueda de catálogo vs. FAQ |

> **Referencia:** [Strands Agents — Tools](https://strandsagents.com/latest/user-guide/concepts/tools/)

In [ ]:
from strands import tool


@tool
def search_catalog(query: str) -> str:
    """Busca productos en el catálogo de TechShop por nombre o descripción.
    Usa esta herramienta cuando el usuario pregunte sobre productos, precios o disponibilidad."""
    catalog_path = DATA_DIR / "catalog.json"
    catalog = json.loads(catalog_path.read_text(encoding="utf-8"))

    query_lower = query.lower()
    matches = []
    for product in catalog:
        searchable = f"{product['nombre']} {product['descripcion']}".lower()
        if query_lower in searchable:
            matches.append({
                "nombre": product["nombre"],
                "precio": product["precio"],
                "stock": product["stock"],
                "descripcion": product["descripcion"],
            })

    if not matches:
        return f"No se encontraron productos para: {query}"
    return json.dumps(matches, ensure_ascii=False, indent=2)

print("[OK] Herramienta search_catalog definida")

**Análisis de `search_catalog`:**

1. **`@tool` decorator**: Registra la función como herramienta. Strands genera automáticamente el JSON Schema a partir de la firma y el docstring.
2. **Docstring como instrucción**: El texto `"Busca productos..."` es lo que el LLM lee para decidir **cuándo** usar esta herramienta. Piensa en el docstring como una instrucción dirigida al modelo.
3. **Búsqueda por substring**: Usamos una búsqueda simple (`query_lower in searchable`). Es frágil: "auriculares bluetooth" no encontrará "auricular inalámbrico". En producción usaríamos búsqueda semántica (embeddings).
4. **Retorno JSON string**: Devolvemos un `str` con JSON, no una lista Python. Esto es porque el resultado viaja de vuelta al LLM como texto.

> **Pregunta:** ¿Qué pasaría si el docstring dijera simplemente `"Busca cosas"`? ¿El LLM sabría cuándo usarla?

In [ ]:
@tool
def get_faq_answer(question: str) -> str:
    """Busca respuestas en las preguntas frecuentes de TechShop.
    Usa esta herramienta para consultas sobre envíos, devoluciones, garantías, pagos u horarios."""
    faqs_path = DATA_DIR / "faqs.json"
    faqs = json.loads(faqs_path.read_text(encoding="utf-8"))

    # Búsqueda simple por palabras clave
    stopwords = {"el", "la", "los", "las", "de", "del", "y", "o", "un", "una", "que", "en"}
    words = [w for w in question.lower().split() if w not in stopwords]

    for faq in faqs:
        searchable = f"{faq['pregunta']} {faq['respuesta']}".lower()
        if any(word in searchable for word in words):
            return json.dumps(faq, ensure_ascii=False, indent=2)

    return f"No se encontró información sobre: {question}"

print("[OK] Herramienta get_faq_answer definida")

**Análisis de `get_faq_answer`:**

- La **búsqueda por palabras clave** con filtrado de stopwords es una heurística simple. Nótese que devuelve solo la **primera coincidencia**, lo cual puede perder resultados relevantes.
- En la versión empaquetada (`src/techshop_agent/tools.py`), verás una implementación más robusta con normalización de acentos (Unicode NFKD) y búsqueda fuzzy con `difflib.SequenceMatcher`.

> **Patrón clave:** En el notebook construimos versiones simplificadas para aprender. El código real está en `src/techshop_agent/` con manejo de errores, normalización y tests. Esto refleja un flujo profesional: **prototipo en notebook -> código en paquete -> tests -> producción**.

---

## Parte 5: Crear el agente TechShop

### La fórmula del agente

$$\text{Agente} = \text{Modelo (LLM)} + \text{Herramientas (tools)} + \text{System Prompt (instrucciones)}$$

El **system prompt** es posiblemente el componente más importante. Define:
- **Identidad**: quién es el agente (nombre, rol, tono)
- **Alcance**: qué puede y qué no puede hacer
- **Reglas**: instrucciones explícitas sobre comportamiento
- **Uso de herramientas**: cuándo y cómo usar las herramientas disponibles

### Principios para diseñar un buen system prompt

| Principio | Ejemplo correcto | Ejemplo débil |
|-----------|------------------|---------------|
| **Ser específico** | "Responde solo sobre productos de TechShop" | "Sé útil" |
| **Dar instrucciones explícitas** | "SIEMPRE usa las herramientas antes de responder" | "Usa herramientas si quieres" |
| **Definir límites** | "Si no encuentras info, dilo honestamente" | (dejar que invente) |
| **Rol concreto** | "Eres un asistente de atención al cliente" | "Eres una IA" |

> **Referencia:** [Anthropic — System Prompts Guide](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/system-prompts) · [Strands Agents — Agent Configuration](https://strandsagents.com/latest/user-guide/concepts/agents/)

In [ ]:
from strands import Agent

SYSTEM_PROMPT = """Eres un asistente de atención al cliente para TechShop, una tienda online de electrónica.

Tu trabajo es ayudar a los clientes con:
- Consultas sobre productos (precios, disponibilidad, especificaciones)
- Preguntas frecuentes (envíos, devoluciones, garantías, pagos, horarios)

Reglas:
- SIEMPRE usa las herramientas disponibles para buscar información antes de responder
- Si no encuentras la información, dilo honestamente
- Responde en español, de forma concisa y profesional
- No inventes información sobre productos o políticas
"""

agent = Agent(
    model=model,
    tools=[search_catalog, get_faq_answer],
    system_prompt=SYSTEM_PROMPT,
)

print("[OK] Agente TechShop creado y listo")

**¿Qué hace el constructor `Agent()`?**

La clase [`Agent`](https://strandsagents.com/latest/user-guide/concepts/agents/) de Strands Agents orquesta el **agentic loop** (bucle agéntico):

1. Recibe la query del usuario
2. Envía al modelo: system prompt + historial + herramientas disponibles
3. Si el modelo responde con un `tool_use`, ejecuta la herramienta y envía el resultado de vuelta (vuelve al paso 2)
4. Cuando el modelo responde con texto final, lo devuelve

Parámetros que hemos usado:
- **`model`**: la instancia de `BedrockModel` con la configuración de inferencia
- **`tools`**: lista de funciones decoradas con `@tool` — sus esquemas se envían al modelo
- **`system_prompt`**: las instrucciones que el modelo recibe como contexto persistente

> **Concepto LLMOps:** El system prompt es un **artefacto versionable**. En el Notebook 3 veremos cómo versionarlo y gestionarlo con Langfuse Prompt Management, exactamente como versionamos código con Git.

---

## Parte 6: Probemos el agente

Vamos a enviar **cuatro tipos de consultas** para explorar el comportamiento del agente:

| Tipo | Qué prueba | Ejemplo |
|------|-----------|---------|
| **Producto directo** | ¿Busca bien en el catálogo? | "¿Qué portátiles tenéis?" |
| **FAQ directo** | ¿Encuentra la respuesta en FAQs? | "¿Política de devoluciones?" |
| **Fuera de ámbito** | ¿Qué hace con preguntas que NO debería responder? | "¿Capital de Francia?" |
| **Ambigua** | ¿Puede inferir la intención y buscar correctamente? | "Algo para escuchar música corriendo" |

> **Mentalidad LLMOps:** No pruebes solo el *happy path*. Los casos extremos, ambiguos y fuera de ámbito son los que fallan en producción.

In [ ]:
# Consulta sobre productos
print("=" * 70)
print("CONSULTA 1: Productos")
print("=" * 70)
response = agent("¿Qué portátiles tenéis disponibles?")
print(response)

In [ ]:
# Consulta sobre FAQs
print("=" * 70)
print("CONSULTA 2: Políticas de la tienda")
print("=" * 70)
response = agent("¿Cuál es la política de devoluciones?")
print(response)

In [ ]:
# Consulta fuera de ámbito
print("=" * 70)
print("CONSULTA 3: Fuera de ámbito")
print("=" * 70)
response = agent("¿Cuál es la capital de Francia?")
print(response)

In [ ]:
# Consulta ambigua
print("=" * 70)
print("CONSULTA 4: Búsqueda que puede fallar")
print("=" * 70)
response = agent("Necesito algo para escuchar música mientras corro")
print(response)

### Análisis de los resultados

Después de ejecutar las 4 consultas, analiza críticamente:

1. **Consulta de producto:** ¿El agente usó `search_catalog`? ¿Devolvió información correcta del catálogo?
2. **Consulta de FAQ:** ¿Usó `get_faq_answer`? ¿La respuesta coincide con los datos de `faqs.json`?
3. **Fuera de ámbito:** ¿Respondió sobre Francia? Si lo hizo, ¿es un problema? ¿Cómo lo solucionaríamos?
4. **Ambigua:** ¿Supo interpretar "algo para escuchar música corriendo" y buscar auriculares?

> **Observación clave:** Sin herramientas de observabilidad, **no podemos confirmar** si el agente realmente llamó a las herramientas o inventó la respuesta. Solo vemos el texto final. Esto es exactamente lo que resolveremos en el Notebook 2.

---

## Ejercicios opcionales

Estos ejercicios te invitan a experimentar con el agente que acabamos de construir. No hay una respuesta "correcta" — el objetivo es que desarrolles intuición sobre el comportamiento de los agentes.

### Ejercicio 1: Experimenta con la temperatura

Modifica la temperatura del modelo y observa cómo cambia el comportamiento. Ejecuta la misma consulta 3 veces con cada configuración.

> **Hipótesis a validar:** ¿Con temperature=0.0 las respuestas son idénticas? ¿Con temperature=1.0 inventa más?

In [ ]:
# === EJERCICIO 1: Experimenta con la temperatura ===
# Descomenta y ejecuta este bloque para comparar comportamientos

# from strands import Agent
# from strands.models import BedrockModel

# test_query = "Recomiéndame algo para trabajar desde casa"

# for temp in [0.0, 0.5, 1.0]:
#     print(f"\n{'=' * 60}")
#     print(f"  Temperature = {temp}")
#     print(f"{'=' * 60}")
#     temp_model = BedrockModel(
#         model_id=os.getenv("MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0"),
#         region_name=os.getenv("AWS_REGION", "us-east-1"),
#         temperature=temp,
#         max_tokens=512,
#     )
#     temp_agent = Agent(
#         model=temp_model,
#         tools=[search_catalog, get_faq_answer],
#         system_prompt=SYSTEM_PROMPT,
#     )
#     response = temp_agent(test_query)
#     print(str(response)[:300])

### Ejercicio 2: Crea tu propia herramienta

Crea una herramienta `get_store_hours` que devuelva los horarios de TechShop. Luego crea un nuevo agente que la incluya y prueba con "¿A qué hora cerráis?".

> **Pista:** Sigue el patrón de `search_catalog` — decora con `@tool`, escribe un docstring descriptivo, y devuelve un string.

In [ ]:
# === EJERCICIO 2: Crea tu propia herramienta ===
# Descomenta, completa la función, y pruébala

# @tool
# def get_store_hours() -> str:
#     """..."""  # <-- Escribe un docstring que ayude al LLM a saber cuándo usar esta tool
#     hours = {
#         "lunes_a_viernes": "9:00 - 21:00",
#         "sabado": "10:00 - 14:00",
#         "domingo": "Cerrado",
#     }
#     return json.dumps(hours, ensure_ascii=False)
#
# agent_with_hours = Agent(
#     model=model,
#     tools=[search_catalog, get_faq_answer, get_store_hours],
#     system_prompt=SYSTEM_PROMPT,
# )
# response = agent_with_hours("¿A qué hora cerráis los sábados?")
# print(response)

### Ejercicio 3: Modifica el system prompt

Crea dos variaciones del system prompt y compara las respuestas a la misma pregunta:

- **Variación A (restrictiva):** Añade "NUNCA respondas preguntas que no sean sobre TechShop. Si te preguntan algo fuera de ámbito, di exactamente: 'Solo puedo ayudarte con consultas sobre TechShop.'"
- **Variación B (conversacional):** Añade "Puedes hacer small talk breve pero siempre redirige al cliente hacia productos o servicios de TechShop."

Prueba ambas con: `"Hola, ¿qué tiempo hace hoy? Por cierto, ¿tenéis tablets?"`

> **Reflexión LLMOps:** El system prompt cambia radicalmente el comportamiento del agente. ¿Cómo decidimos cuál es "mejor"? -> Esto es exactamente lo que haremos con **evaluaciones** en el Notebook 4.

In [ ]:
# === EJERCICIO 3: Variaciones de system prompt ===
# Descomenta y ejecuta para comparar

# PROMPT_RESTRICTIVO = SYSTEM_PROMPT + """
# NUNCA respondas preguntas que no sean sobre TechShop.
# Si te preguntan algo fuera de ámbito, di exactamente:
# 'Solo puedo ayudarte con consultas sobre TechShop.'
# """

# PROMPT_CONVERSACIONAL = SYSTEM_PROMPT + """
# Puedes hacer small talk breve y amigable, pero siempre redirige
# al cliente hacia productos o servicios de TechShop.
# """

# test_query = "Hola, ¿qué tiempo hace hoy? Por cierto, ¿tenéis tablets?"

# for label, prompt in [("RESTRICTIVO", PROMPT_RESTRICTIVO), ("CONVERSACIONAL", PROMPT_CONVERSACIONAL)]:
#     print(f"\n{'=' * 60}")
#     print(f"  Prompt: {label}")
#     print(f"{'=' * 60}")
#     test_agent = Agent(model=model, tools=[search_catalog, get_faq_answer], system_prompt=prompt)
#     print(str(test_agent(test_query))[:400])

---

## Parte 7: Del notebook al paquete Python

Todo lo que construimos paso a paso ya existe empaquetado en `src/techshop_agent/`:

| Archivo | Contenido |
|---------|-----------|
| `agent.py` | Función `create_agent()` — el orquestador |
| `tools.py` | `search_catalog` y `get_faq_answer` con búsqueda robusta |
| `config.py` | `SYSTEM_PROMPT`, loaders de datos, configuración |
| `guardrails.py` | Validación de input/output (lo veremos en NB 6) |

A partir del **Notebook 2** importaremos el agente directamente:

```python
from techshop_agent import create_agent
agent = create_agent()
```

> **Patrón profesional:** Prototipa en notebooks, empaqueta en módulos con tests. Nunca pongas un notebook en producción.

---

## Resumen: lo que hemos construido

| Componente | Estado |
|-----------|--------|
| Agente TechShop (Strands + Bedrock + Tools) | Funcional |
| search_catalog + get_faq_answer | Definidos y probados |
| System prompt con boundaries | Hardcodeado (lo versionaremos en NB 3) |

### Conceptos clave

- **Agente = LLM + Tools + System Prompt** -- el LLM razona, las tools ejecutan
- **Strands Agents** usa un enfoque model-driven: el LLM controla el flujo
- **Tool calling** funciona a traves de JSON Schema generado desde el decorador `@tool`
- **El system prompt** es un artefacto critico que debe versionarse y evaluarse
- **Observabilidad desde el inicio** -- no se anade despues, se instrumenta mientras se desarrolla

### Roadmap del curso

```mermaid
flowchart LR
    NB1["NB 1 (HOY)<br/>Setup y Agente"] -->|Dia 1| NB2["NB 2<br/>Observabilidad"]
    NB2 --> NB3["NB 3<br/>Prompt Management"]
    NB3 -->|Dia 2| NB4["NB 4<br/>Evaluacion"]
    NB4 --> NB5["NB 5<br/>CI/CD Gate"]
    NB5 -->|Dia 3| NB6["NB 6<br/>Guardrails"]
    NB6 --> NB7["NB 7<br/>Pipeline Integrado"]

    style NB1 fill:#2d5016,stroke:#4a8c28,color:#fff
```

> **El objetivo del curso no es construir el agente -- es hacerlo operable, observable, evaluable y seguro.**

---

### Referencias del notebook

- [Strands Agents SDK -- GitHub](https://github.com/strands-agents/sdk-python)
- [Strands Agents -- Documentacion oficial](https://strandsagents.com/latest/)
- [Strands Agents -- Tools](https://strandsagents.com/latest/user-guide/concepts/tools/)
- [Amazon Bedrock -- Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html)
- [Anthropic -- Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents)

---

## Siguiente: [Notebook 2 -- Observabilidad con Langfuse](./02_observability.ipynb)